# Your First LLM-Powered Tool

Starter notebook. Fill in the TODOs and **run every cell** before committing.
Do **not** commit your API key — load it from `.env`.


In [ ]:
import os, json, re
from google import genai
from google.genai import types
from google.api_core.exceptions import ResourceExhausted
import time

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY (see .env)"
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-3.5-flash"


## Task 1 — First calls and the sampling dial

Make a working call (print response **and token usage**), then send the same prompt 3× at temp ~0.1 and 3× at temp ~1.0.


In [ ]:
SYSTEM = "You are a concise support assistant."
TICKET = (
    "Hey, I was charged twice this month for my subscription. "
    "Can someone look into this?"
)
 
response = client.models.generate_content(
    model=MODEL,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM,
        temperature=0.5,
    ),
    contents=TICKET,
)
 
print("=" * 60)
print("RESPONSE")
print("=" * 60)
print(response.text)
 
print("\n" + "=" * 60)
print("TOKEN USAGE")
print("=" * 60)
u = response.usage_metadata
print(f"  prompt tokens   : {u.prompt_token_count}")
print(f"  response tokens : {u.candidates_token_count}")
print(f"  total           : {u.total_token_count}")
 
CRASH_TICKET = (
    "The dashboard keeps crashing every time I try to export a report. "
    "This has been happening since the last update."
)
 
for temp in [0.1, 1.0]:
    print("\n" + "#" * 60)
    print(f"  TEMPERATURE = {temp}")
    print("#" * 60)
    for run in range(1, 4):
        r = client.models.generate_content(
            model=MODEL,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM,
                temperature=temp,
            ),
            contents=CRASH_TICKET,
        )
        print(f"\n  run {run}: {r.text.strip()}")


**What changed, and why?**

> At temperature 0.1 all three runs returned identical replies word-for-word;
at 1.0 each run varied in structure, phrasing, and which follow-up questions
were asked. This is because temperature rescales the model's logit
distribution before sampling — a near-zero value sharpens it so the top token
wins almost every time, while 1.0 leaves it untouched so lower-probability
tokens get occasionally picked, cascading into different outputs. For a
classifier that must return a consistent label, low temperature is the right
default; high temperature suits tasks where variety is a feature.


## Task 2 — Prompt-pattern bake-off

Classify the tickets three ways (zero-shot, few-shot, chain-of-thought) into `billing / bug / feature_request / other`. Build a comparison table. Record prompts + verdict in `prompts.md`.


In [ ]:
MODEL = "gemini-3.5-flash"

with open("tickets.json") as f:
    tickets = json.load(f)

LABELS = ["billing", "bug", "feature_request", "other"]

SYSTEM_INSTRUCTION = f"""You'll build a small support-ticket classifier: given a short customer message, label it as one of billing, bug, feature_request, or other. You'll attack it three ways and end with a structured, parseable version."""

def classify(text, style):
    """
    Classifies a ticket text using Gemini 2.5 Flash based on the requested style.
    """
    # 1. Build the prompt based on the style
    if style == "zero_shot":
        prompt = f"""
        Classify the following customer support ticket.
        
        Ticket:
        "{text}"
        
        Provide your answer as a single word from the allowed categories list. Do not include any other text.
        Category:
        """
        
    elif style == "few_shot":
        prompt = f"""
        Classify the support ticket into one of the allowed categories. Here are some examples:
        
        Ticket: "I can't seem to find where to update my credit card information on the dashboard."
        Category: billing
        
        Ticket: "The app crashes every single time I try to export my monthly report to PDF."
        Category: bug
        
        Ticket: "It would be amazing if we could get a dark mode option for the mobile app."
        Category: feature_request
        
        Ticket: "Can you please delete my account? I no longer need this service."
        Category: other
        
        Now classify this ticket:
        Ticket: "{text}"
        Category:
        """
        
    elif style == "cot":
        prompt = f"""
        Analyze the following customer support ticket step-by-step before selecting the final category.
        
        Ticket: "{text}"
        
        Please provide your response in the following format:
        <thinking>
        [Your step-by-step reasoning here]
        </thinking>
        <category>[The final category word here]</category>
        """
    else:
        raise ValueError(f"Unknown style: {style}")

    # 2. Call the model
    # Using gemini-2.5-flash as it is fast, highly capable, and cost-effective for classification tasks.
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            # Lower temperature encourages more deterministic and exact label matching
            temperature=0.1, 
        ),
    )
    
    result = response.text.strip().lower()
    
    # 3. Post-process the result to return a clean label
    if style == "cot":
        # Extract content between <category> tags for Chain of Thought
        if "<category>" in result and "</category>" in result:
            result = result.split("<category>")[1].split("</category>")[0].strip()
    
    # Clean up any potential accidental punctuation or formatting
    for label in LABELS:
        if label in result:
            return label
            
    return "unknown"

# --- Run all three styles and assemble a results table ---

print(f"{'Ticket Preview':<40} | {'Zero-Shot':<15} | {'Few-Shot':<15} | {'CoT':<15}")
print("-" * 95)

results = []
for ticket in tickets:
    text = ticket.get("text", "")
    # Cleaned up preview logic
    preview = text if len(text) <= 37 else text[:37] + "..."
    
    # Run predictions
    zero_shot_res = classify(text, "zero_shot")
    few_shot_res = classify(text, "few_shot")
    cot_res = classify(text, "cot")
    
    # Print row in real-time
    print(f"{preview:<40} | {zero_shot_res:<15} | {few_shot_res:<15} | {cot_res:<15}")
    
    # Store results for downstream analysis
    results.append({
        "ticket": text,
        "zero_shot": zero_shot_res,
        "few_shot": few_shot_res,
        "cot": cot_res
    })

## Task 3 — Structured output as a function

Return JSON `{label, confidence, reason}` with low temperature, then **parse and validate** it. Handle one bad-output case. Re-run against local Ollama and compare.


In [ ]:
import os, json, time, re
from google import genai
from google.genai import types
from openai import OpenAI
from google.api_core.exceptions import ResourceExhausted
 
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL  = "gemini-2.5-flash"
 
tickets = json.load(open("tickets.json"))
LABELS  = ["billing", "bug", "feature_request", "other"]
 
SCHEMA = {
    "type": "object",
    "properties": {
        "label":      {"type": "string", "enum": LABELS},
        "confidence": {"type": "number"},
        "reason":     {"type": "string"},
    },
    "required": ["label", "confidence", "reason"],
}
 
PROMPT = lambda t: (
    f"Classify this support ticket.\n"
    f"Ticket: {t}\n\n"
    f"Return JSON with keys: label (one of {LABELS}), "
    f"confidence (0-1), reason (one sentence)."
)
 
FALLBACK = {"label": "other", "confidence": 0.0, "reason": "parse error"}
 
def validate(data: dict) -> dict:
    assert data["label"] in LABELS,                       "bad label"
    assert 0.0 <= float(data["confidence"]) <= 1.0,       "bad confidence"
    assert isinstance(data.get("reason", ""), str),        "bad reason"
    return data
 
def classify_structured(text: str) -> dict:
    for attempt in range(4):
        try:
            r = client.models.generate_content(
                model=MODEL,
                config=types.GenerateContentConfig(
                    system_instruction="You are a support-ticket classifier. Reply only with valid JSON.",
                    temperature=0.1,
                    response_mime_type="application/json",
                    response_schema=SCHEMA,
                ),
                contents=PROMPT(text),
            )
            return validate(json.loads(r.text))
        except ResourceExhausted as e:
            if attempt == 3: raise
            wait = float((re.search(r"(\d+)s", str(e)) or [0, 62])[1]) + 2
            print(f"429 — sleeping {wait:.0f}s"); time.sleep(wait)
        except (json.JSONDecodeError, AssertionError, KeyError) as e:
            print(f"  ⚠ malformed output ({e}) — returning fallback")
            return {**FALLBACK, "reason": str(e)}

 
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
OLLAMA_MODEL = "llama3.2"
 
def classify_ollama(text: str) -> dict:
    try:
        r = ollama.chat.completions.create(
            model=OLLAMA_MODEL,
            temperature=0.1,
            messages=[
                {"role": "system", "content": "You are a support-ticket classifier. Reply only with valid JSON, no markdown."},
                {"role": "user",   "content": PROMPT(text)},
            ],
        )
        raw = r.choices[0].message.content.strip()
        raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
        return validate(json.loads(raw))
    except (json.JSONDecodeError, AssertionError, KeyError) as e:
        print(f"  ⚠ Ollama malformed ({e}): {raw[:120]!r}")
        return {**FALLBACK, "reason": str(e)}
    except Exception as e:
        print(f"  ⚠ Ollama error ({e})")
        return {**FALLBACK, "reason": str(e)}

 
results = []
for ticket in tickets:
    gemini = classify_structured(ticket["text"]); time.sleep(5)
    ollama_result = classify_ollama(ticket["text"])
    results.append({"id": ticket["id"], "text": ticket["text"],
                    "gemini": gemini, "ollama": ollama_result})

 
W = 16
print(f"{'ID':<6}{'GEMINI':<{W}}{'CONF':>6}  {'OLLAMA':<{W}}{'CONF':>6}  TEXT")
print("-" * 85)
for r in results:
    g, o = r["gemini"], r["ollama"]
    valid_g = "✓" if g["label"] != "other" or g["confidence"] > 0 else "✗"
    valid_o = "✓" if o["label"] != "other" or o["confidence"] > 0 else "✗"
    print(
        f"{r['id']:<6}"
        f"{g['label']:<{W}}{g['confidence']:>5.2f}{valid_g} "
        f"{o['label']:<{W}}{o['confidence']:>5.2f}{valid_o} "
        f"{r['text'][:45]}"
    )
 
json.dump(results, open("results_structured.json", "w"), indent=2)
print("\n✓ saved results_structured.json")


**Local vs hosted: did the small local model produce valid JSON as reliably?**

> Gemini returned valid JSON on every call because constrained decoding enforces
the schema at the token level, leaving no room for malformed output. The local
model followed the format most of the time but occasionally wrapped the JSON
in markdown fences or dropped a required key, requiring the fallback handler.
